In [1]:
import pandas as pd
import os

# 1. Setup paths and Chunking
input_file = 'C:\\my-dublin-bikes-project\\notebooks\\bikes_raw_merged.csv'
output_file = 'C:\\my-dublin-bikes-project\\outputs\\bikes_cleaned.csv'
chunk_size = 100000 

# Use the same efficient dtypes as before
dtypes = {
    'STATION ID': 'int16', 'BIKE_STANDS': 'int16',
    'AVAILABLE_BIKE_STANDS': 'int16', 'AVAILABLE_BIKES': 'int16',
    'STATUS': 'category', 'ADDRESS': 'category',
    'LATITUDE': 'float32', 'LONGITUDE': 'float32'
}

In [2]:
# 2. Start Cleaning
if os.path.exists(output_file):
    os.remove(output_file)

print("Starting memory-safe cleaning...")

is_first_chunk = True

# Process the giant file in manageable "bites"
for chunk in pd.read_csv(input_file, dtype=dtypes, parse_dates=['TIME'], chunksize=chunk_size):
    
    # A. Drop complete duplicates within this chunk
    chunk = chunk.drop_duplicates()
    
# B. Fill missing values
    # We replace missing bikes with 0, and use forward/backward fill for stands
    chunk['AVAILABLE_BIKES'] = chunk['AVAILABLE_BIKES'].fillna(0)
    
    # Using the new dedicated functions instead of fillna(method=...)
    chunk['BIKE_STANDS'] = chunk['BIKE_STANDS'].ffill().bfill()
    
    # C. Constraints: Ensure bikes don't exceed stands
    chunk['AVAILABLE_BIKES'] = chunk['AVAILABLE_BIKES'].clip(lower=0, upper=chunk['BIKE_STANDS'])
    
    # D. Calculate Occupancy Rate
    # We use a small epsilon (1e-6) to avoid division by zero errors
    chunk['occupancy_rate'] = (chunk['AVAILABLE_BIKES'] / chunk['BIKE_STANDS'].replace(0, 1)).clip(0, 1)
    
    # E. Save this cleaned chunk
    if is_first_chunk:
        chunk.to_csv(output_file, index=False, mode='w')
        is_first_chunk = False
    else:
        chunk.to_csv(output_file, index=False, mode='a', header=False)

print(f"✅ Success! Cleaned data saved as {output_file}")

Starting memory-safe cleaning...
✅ Success! Cleaned data saved as C:\my-dublin-bikes-project\outputs\bikes_cleaned.csv
